In [0]:
from datetime import datetime, timezone

dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("batch_id", "2025-01-15")
dbutils.widgets.dropdown("run_mode", "initial", ["initial", "incremental"])
dbutils.widgets.text("job_run_id", "")

environment = dbutils.widgets.get("environment")
batch_id = dbutils.widgets.get("batch_id")
run_mode = dbutils.widgets.get("run_mode")
job_run_id = dbutils.widgets.get("job_run_id")

pipeline_name = "education_qa_pipeline"

if job_run_id:
    run_id = f"{environment}_{pipeline_name}_{batch_id}_{run_mode}_job_{job_run_id}"
else:
    run_timestamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    run_id = f"{environment}_{pipeline_name}_{batch_id}_{run_mode}_{run_timestamp}"

catalog = "dbw_edu_qa_dev"

print(f"environment: {environment}")
print(f"batch_id: {batch_id}")
print(f"run_mode: {run_mode}")
print(f"job_run_id: {job_run_id}")
print(f"run_id: {run_id}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.reporting")

spark.sql(f"SHOW SCHEMAS IN {catalog}").show(truncate=False)

environment: dev
batch_id: 2026-01-15
run_mode: incremental
job_run_id: 
run_id: dev_education_qa_pipeline_2026-01-15_incremental_20260608T054450Z
+------------------+
|databaseName      |
+------------------+
|bronze            |
|default           |
|gold              |
|information_schema|
|qa                |
|reporting         |
|silver            |
+------------------+



### vw_data_quality_rule_detail, vw_data_quality_summary

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_data_quality_rule_detail AS
SELECT
    dq.batch_id,
    b.batch_label,

    dq.rule_id,
    dq.rule_name,
    dq.target_table,
    dq.severity,
    dq.severity_sort_order,

    dq.status,
    dq.status_sort_order,

    dq.failed_record_count,
    dq.issue_flag,

    dq.run_id,
    dq.run_timestamp,
    dq.gold_load_timestamp
FROM {catalog}.gold.fact_data_quality_result dq
LEFT JOIN {catalog}.gold.dim_batch b
    ON dq.batch_key = b.batch_key
""")

DataFrame[]

In [0]:
display(
    spark.table(f"{catalog}.reporting.vw_data_quality_rule_detail")
    .orderBy("batch_id", "rule_id")
)


batch_id,batch_label,rule_id,rule_name,target_table,severity,severity_sort_order,status,status_sort_order,failed_record_count,issue_flag,run_id,run_timestamp,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,DQ001,Missing student ID,silver.students,High,1,FAIL,1,1,1,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:58:42.841Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ002,Missing school ID,silver.students,High,1,FAIL,1,1,1,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:58:47.640Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ003,Invalid attendance days,silver.attendance,High,1,FAIL,1,1,1,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:58:51.894Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ004,Duplicate attendance business record,silver.attendance,Medium,2,FAIL,1,4,1,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:58:56.381Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ005,Attendance references missing student,silver.attendance,High,1,FAIL,1,1,1,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:59:01.798Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ006,Assessment references missing student,silver.assessment_results,High,1,PASS,3,0,0,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:59:07.644Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ007,Invalid assessment score,silver.assessment_results,High,1,FAIL,1,1,1,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:59:10.213Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ008,Invalid proficiency band,silver.assessment_results,Medium,2,PASS,3,0,0,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:59:14.775Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ009,Invalid school status,silver.schools,Medium,2,PASS,3,0,0,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:59:17.095Z,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,DQ010,Future attendance month,silver.attendance,Medium,2,PASS,3,0,0,dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,2026-06-08T04:59:19.350Z,2026-06-08T05:41:40.235Z


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_data_quality_summary AS
SELECT
    dq.batch_id,
    b.batch_label,

    dq.severity,
    dq.severity_sort_order,

    dq.status,
    dq.status_sort_order,

    COUNT(*) AS rule_count,
    SUM(dq.failed_record_count) AS failed_record_count,
    SUM(dq.issue_flag) AS issue_rule_count,

    MAX(dq.gold_load_timestamp) AS gold_load_timestamp
FROM {catalog}.gold.fact_data_quality_result dq
LEFT JOIN {catalog}.gold.dim_batch b
    ON dq.batch_key = b.batch_key
GROUP BY
    dq.batch_id,
    b.batch_label,
    dq.severity,
    dq.severity_sort_order,
    dq.status,
    dq.status_sort_order
""")


DataFrame[]

In [0]:
display(
    spark.table(f"{catalog}.reporting.vw_data_quality_summary")
    .orderBy("batch_id", "severity_sort_order", "status_sort_order")
)

batch_id,batch_label,severity,severity_sort_order,status,status_sort_order,rule_count,failed_record_count,issue_rule_count,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,High,1,FAIL,1,5,5,5,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,High,1,PASS,3,1,0,0,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,Medium,2,FAIL,1,1,4,1,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,Medium,2,PASS,3,3,0,0,2026-06-08T05:41:40.235Z
2025-01-15,Batch 1 - 2024 data,Low,3,WARN,2,1,5,1,2026-06-08T05:41:40.235Z
2026-01-15,Batch 2 - 2025 data,High,1,FAIL,1,4,4,4,2026-06-08T05:42:48.100Z
2026-01-15,Batch 2 - 2025 data,High,1,PASS,3,2,0,0,2026-06-08T05:42:48.100Z
2026-01-15,Batch 2 - 2025 data,Medium,2,FAIL,1,1,4,1,2026-06-08T05:42:48.100Z
2026-01-15,Batch 2 - 2025 data,Medium,2,PASS,3,3,0,0,2026-06-08T05:42:48.100Z
2026-01-15,Batch 2 - 2025 data,Low,3,PASS,3,1,0,0,2026-06-08T05:42:48.100Z


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_defect_log AS
SELECT
    d.batch_id,
    b.batch_label,

    d.rule_id,
    d.rule_name,
    d.target_table,

    d.severity,
    d.severity_sort_order,

    d.defect_id,
    d.defect_title,
    d.defect_status,
    d.defect_status_sort_order,
    d.open_defect_flag,

    d.failed_record_count,
    d.recommended_action,
    d.created_timestamp,
    d.gold_load_timestamp
FROM {catalog}.gold.fact_defect d
LEFT JOIN {catalog}.gold.dim_batch b
    ON d.batch_key = b.batch_key
""")



DataFrame[]

In [0]:
display(
    spark.table(f"{catalog}.reporting.vw_defect_log")
    .orderBy("batch_id", "severity_sort_order", "rule_id")
)

batch_id,batch_label,rule_id,rule_name,target_table,severity,severity_sort_order,defect_id,defect_title,defect_status,defect_status_sort_order,open_defect_flag,failed_record_count,recommended_action,created_timestamp,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,DQ001,Missing student ID,silver.students,High,1,DQ001_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,Missing student ID,Open,1,1,1,Investigate source student extract and require student_id for all student records.,2026-06-08T04:58:44.629Z,2026-06-08T05:41:47.027Z
2025-01-15,Batch 1 - 2024 data,DQ002,Missing school ID,silver.students,High,1,DQ002_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,Missing school ID,Open,1,1,1,Investigate student enrolment extract and require a valid school_id for each student.,2026-06-08T04:58:49.334Z,2026-06-08T05:41:47.027Z
2025-01-15,Batch 1 - 2024 data,DQ003,Invalid attendance days,silver.attendance,High,1,DQ003_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,Invalid attendance days,Open,1,1,1,Review attendance source extract and enforce valid possible_days and attended_days values before publishing.,2026-06-08T04:58:53.592Z,2026-06-08T05:41:47.027Z
2025-01-15,Batch 1 - 2024 data,DQ005,Attendance references missing student,silver.attendance,High,1,DQ005_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,Attendance references missing student,Open,1,1,1,Investigate attendance records with missing student references and correct source system referential integrity.,2026-06-08T04:59:04.440Z,2026-06-08T05:41:47.027Z
2025-01-15,Batch 1 - 2024 data,DQ007,Invalid assessment score,silver.assessment_results,High,1,DQ007_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,Invalid assessment score,Open,1,1,1,Review assessment scoring scale and source extract. Confirm whether scores should be normalised before reporting.,2026-06-08T04:59:11.923Z,2026-06-08T05:41:47.027Z
2025-01-15,Batch 1 - 2024 data,DQ004,Duplicate attendance business record,silver.attendance,Medium,2,DQ004_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,Duplicate attendance business record,Open,1,1,4,Review attendance source extract for duplicate monthly attendance submissions and deduplicate before reporting.,2026-06-08T04:58:58.668Z,2026-06-08T05:41:47.027Z
2025-01-15,Batch 1 - 2024 data,DQ011,School event linked to inactive or missing school,silver.school_events,Low,3,DQ011_2025-01-15_dev_education_qa_pipeline_2025-01-15_initial_20260608T045833Z,School event linked to inactive or missing school,Open,1,1,5,Review school event source data and confirm whether events should be linked only to active schools.,2026-06-08T04:59:23.700Z,2026-06-08T05:41:47.027Z
2026-01-15,Batch 2 - 2025 data,DQ002,Missing school ID,silver.students,High,1,DQ002_2026-01-15_dev_education_qa_pipeline_2026-01-15_incremental_20260608T045942Z,Missing school ID,Open,1,1,1,Investigate student enrolment extract and require a valid school_id for each student.,2026-06-08T04:59:55.035Z,2026-06-08T05:42:52.309Z
2026-01-15,Batch 2 - 2025 data,DQ003,Invalid attendance days,silver.attendance,High,1,DQ003_2026-01-15_dev_education_qa_pipeline_2026-01-15_incremental_20260608T045942Z,Invalid attendance days,Open,1,1,1,Review attendance source extract and enforce valid possible_days and attended_days values before publishing.,2026-06-08T04:59:59.605Z,2026-06-08T05:42:52.309Z
2026-01-15,Batch 2 - 2025 data,DQ005,Attendance references missing student,silver.attendance,High,1,DQ005_2026-01-15_dev_education_qa_pipeline_2026-01-15_incremental_20260608T045942Z,Attendance references missing student,Open,1,1,1,Investigate attendance records with missing student references and correct source system referential integrity.,2026-06-08T05:00:10.874Z,2026-06-08T05:42:52.309Z


In [0]:
display(
    spark.table(f"{catalog}.reporting.vw_data_quality_rule_detail")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

display(
    spark.table(f"{catalog}.reporting.vw_defect_log")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

batch_id,count
2025-01-15,11
2026-01-15,11


batch_id,count
2025-01-15,7
2026-01-15,5


### vw_attendance_by_school_month,  vw_attendance_by_year_level

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_attendance_by_school_month AS
SELECT
    f.batch_id,
    b.batch_label,

    s.school_id,
    s.school_name,
    s.region,
    s.school_type,
    s.status AS school_status,

    d.full_date AS attendance_month,
    d.calendar_year AS attendance_year,
    d.month_number AS attendance_month_number,
    d.month_name AS attendance_month_name,

    SUM(f.possible_days) AS possible_days,
    SUM(f.attended_days) AS attended_days,
    COUNT(DISTINCT f.student_batch_key) AS student_count,

    CASE
        WHEN SUM(f.possible_days) > 0
        THEN ROUND(SUM(f.attended_days) / SUM(f.possible_days), 4)
        ELSE NULL
    END AS attendance_rate,

    CASE
        WHEN SUM(f.possible_days) > 0
        THEN ROUND((SUM(f.attended_days) / SUM(f.possible_days)) * 100, 2)
        ELSE NULL
    END AS attendance_rate_percent,

    CASE
        WHEN SUM(f.possible_days) = 0 THEN 'No possible days'
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.80 THEN 'Below 80%'
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.85 THEN '80% to below 85%'
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.90 THEN '85% to below 90%'
        ELSE '90% and above'
    END AS attendance_band,

    CASE
        WHEN SUM(f.possible_days) = 0 THEN 0
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.80 THEN 1
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.85 THEN 2
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.90 THEN 3
        ELSE 4
    END AS attendance_band_sort_order,

    MAX(f.gold_load_timestamp) AS gold_load_timestamp
FROM {catalog}.gold.fact_attendance f
LEFT JOIN {catalog}.gold.dim_batch b
    ON f.batch_key = b.batch_key
LEFT JOIN {catalog}.gold.dim_school s
    ON f.school_scd_key = s.school_scd_key
LEFT JOIN {catalog}.gold.dim_date d
    ON f.attendance_month_date_key = d.date_key
GROUP BY
    f.batch_id,
    b.batch_label,
    s.school_id,
    s.school_name,
    s.region,
    s.school_type,
    s.status,
    d.full_date,
    d.calendar_year,
    d.month_number,
    d.month_name
""")

display(
    spark.table(f"{catalog}.reporting.vw_attendance_by_school_month")
    .orderBy("batch_id", "school_id", "attendance_month")
    .limit(50)
)


batch_id,batch_label,school_id,school_name,region,school_type,school_status,attendance_month,attendance_year,attendance_month_number,attendance_month_name,possible_days,attended_days,student_count,attendance_rate,attendance_rate_percent,attendance_band,attendance_band_sort_order,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-01-01,2024,1,Jan,0,0,200,null,null,No possible days,0,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-02-01,2024,2,Feb,3985,3418,200,0.8577,85.77,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-03-01,2024,3,Mar,3975,3405,200,0.8566,85.66,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-04-01,2024,4,Apr,4015,3387,200,0.8436,84.36,80% to below 85%,2,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-05-01,2024,5,May,3991,3422,200,0.8574,85.74,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-06-01,2024,6,Jun,3981,3340,200,0.839,83.9,80% to below 85%,2,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-07-01,2024,7,Jul,4003,3347,200,0.8361,83.61,80% to below 85%,2,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-08-01,2024,8,Aug,4030,3444,200,0.8546,85.46,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-09-01,2024,9,Sep,3980,3429,200,0.8616,86.16,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,SCH001,ACT Education School 001,North Canberra,High School,Active,2024-10-01,2024,10,Oct,3988,3383,200,0.8483,84.83,80% to below 85%,2,2026-06-08T05:31:37.324Z


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_attendance_by_year_level AS
SELECT
    f.batch_id,
    b.batch_label,

    yl.year_level,
    yl.year_level_label,
    yl.year_level_sort_order,

    d.full_date AS attendance_month,
    d.calendar_year AS attendance_year,
    d.month_number AS attendance_month_number,
    d.month_name AS attendance_month_name,

    SUM(f.possible_days) AS possible_days,
    SUM(f.attended_days) AS attended_days,
    COUNT(DISTINCT f.student_batch_key) AS student_count,

    CASE
        WHEN SUM(f.possible_days) > 0
        THEN ROUND(SUM(f.attended_days) / SUM(f.possible_days), 4)
        ELSE NULL
    END AS attendance_rate,

    CASE
        WHEN SUM(f.possible_days) > 0
        THEN ROUND((SUM(f.attended_days) / SUM(f.possible_days)) * 100, 2)
        ELSE NULL
    END AS attendance_rate_percent,

    CASE
        WHEN SUM(f.possible_days) = 0 THEN 'No possible days'
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.80 THEN 'Below 80%'
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.85 THEN '80% to below 85%'
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.90 THEN '85% to below 90%'
        ELSE '90% and above'
    END AS attendance_band,

    CASE
        WHEN SUM(f.possible_days) = 0 THEN 0
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.80 THEN 1
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.85 THEN 2
        WHEN SUM(f.attended_days) / SUM(f.possible_days) < 0.90 THEN 3
        ELSE 4
    END AS attendance_band_sort_order,

    MAX(f.gold_load_timestamp) AS gold_load_timestamp
FROM {catalog}.gold.fact_attendance f
LEFT JOIN {catalog}.gold.dim_batch b
    ON f.batch_key = b.batch_key
LEFT JOIN {catalog}.gold.dim_year_level yl
    ON f.year_level_key = yl.year_level_key
LEFT JOIN {catalog}.gold.dim_date d
    ON f.attendance_month_date_key = d.date_key
GROUP BY
    f.batch_id,
    b.batch_label,
    yl.year_level,
    yl.year_level_label,
    yl.year_level_sort_order,
    d.full_date,
    d.calendar_year,
    d.month_number,
    d.month_name
""")

display(
    spark.table(f"{catalog}.reporting.vw_attendance_by_year_level")
    .orderBy("batch_id", "year_level_sort_order", "attendance_month")
    .limit(50)
)

batch_id,batch_label,year_level,year_level_label,year_level_sort_order,attendance_month,attendance_year,attendance_month_number,attendance_month_name,possible_days,attended_days,student_count,attendance_rate,attendance_rate_percent,attendance_band,attendance_band_sort_order,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-01-01,2024,1,Jan,0,0,971,null,null,No possible days,0,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-02-01,2024,2,Feb,19410,16505,971,0.8503,85.03,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-03-01,2024,3,Mar,19458,16501,971,0.848,84.8,80% to below 85%,2,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-04-01,2024,4,Apr,19475,16622,971,0.8535,85.35,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-05-01,2024,5,May,19520,16601,971,0.8505,85.05,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-06-01,2024,6,Jun,19414,16489,971,0.8493,84.93,80% to below 85%,2,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-07-01,2024,7,Jul,19315,16503,971,0.8544,85.44,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-08-01,2024,8,Aug,19462,16629,971,0.8544,85.44,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-09-01,2024,9,Sep,19512,16609,971,0.8512,85.12,85% to below 90%,3,2026-06-08T05:31:37.324Z
2025-01-15,Batch 1 - 2024 data,0,Kindergarten,0,2024-10-01,2024,10,Oct,19502,16642,971,0.8533,85.33,85% to below 90%,3,2026-06-08T05:31:37.324Z


In [0]:
display(
    spark.table(f"{catalog}.reporting.vw_attendance_by_school_month")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

display(
    spark.table(f"{catalog}.reporting.vw_attendance_by_year_level")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

batch_id,count
2025-01-15,600
2026-01-15,576


batch_id,count
2025-01-15,156
2026-01-15,156


### vw_assessment_by_school, vw_assessment_by_domain

In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_assessment_by_school AS
SELECT
    f.batch_id,
    b.batch_label,

    d.calendar_year AS assessment_year,

    s.school_id,
    s.school_name,
    s.region,
    s.school_type,
    s.status AS school_status,

    COUNT(*) AS assessment_count,
    COUNT(DISTINCT f.student_batch_key) AS student_count,

    ROUND(AVG(f.score), 2) AS average_score,
    CAST(percentile_approx(f.score, 0.5) AS INT) AS median_score,
    MIN(f.score) AS min_score,
    MAX(f.score) AS max_score,

    CASE
        WHEN AVG(f.score) < 400 THEN 'Below 400'
        WHEN AVG(f.score) < 550 THEN '400 to below 550'
        ELSE '550 and above'
    END AS average_score_band,

    CASE
        WHEN AVG(f.score) < 400 THEN 1
        WHEN AVG(f.score) < 550 THEN 2
        ELSE 3
    END AS average_score_band_sort_order,

    MAX(f.gold_load_timestamp) AS gold_load_timestamp
FROM {catalog}.gold.fact_assessment_result f
LEFT JOIN {catalog}.gold.dim_batch b
    ON f.batch_key = b.batch_key
LEFT JOIN {catalog}.gold.dim_school s
    ON f.school_scd_key = s.school_scd_key
LEFT JOIN {catalog}.gold.dim_date d
    ON f.assessment_year_date_key = d.date_key
GROUP BY
    f.batch_id,
    b.batch_label,
    d.calendar_year,
    s.school_id,
    s.school_name,
    s.region,
    s.school_type,
    s.status
""")

display(
    spark.table(f"{catalog}.reporting.vw_assessment_by_school")
    .orderBy("batch_id", "assessment_year", "school_id")
    .limit(50)
)


batch_id,batch_label,assessment_year,school_id,school_name,region,school_type,school_status,assessment_count,student_count,average_score,median_score,min_score,max_score,average_score_band,average_score_band_sort_order,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,2024,SCH001,ACT Education School 001,North Canberra,High School,Active,600,200,516.1,514,413,620,400 to below 550,2,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH002,ACT Education School 002,North Canberra,Primary,Active,630,210,367.98,368,250,518,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH003,ACT Education School 003,North Canberra,Primary,Active,567,189,372.14,370,250,520,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH004,ACT Education School 004,North Canberra,Primary,Active,513,171,386.18,390,250,515,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH005,ACT Education School 005,Tuggeranong,High School,Active,537,179,513.27,514,412,619,400 to below 550,2,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH006,ACT Education School 006,Weston Creek,Primary,Active,609,203,369.02,367,250,514,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH007,ACT Education School 007,Tuggeranong,Primary,Active,600,200,375.59,371,250,518,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH008,ACT Education School 008,Weston Creek,Primary,Active,591,197,381.87,382,250,516,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH009,ACT Education School 009,Belconnen,Primary,Active,624,208,380.59,382,250,520,Below 400,1,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,SCH010,ACT Education School 010,Woden,College,Active,567,189,590.41,591,510,669,550 and above,3,2026-06-08T05:37:27.731Z


In [0]:
spark.sql(f"""
CREATE OR REPLACE VIEW {catalog}.reporting.vw_assessment_by_domain AS
SELECT
    f.batch_id,
    b.batch_label,

    d.calendar_year AS assessment_year,

    dom.domain,
    dom.domain_sort_order,

    band.proficiency_band,
    band.proficiency_band_sort_order,
    band.score_range_label,

    COUNT(*) AS assessment_count,
    COUNT(DISTINCT f.student_batch_key) AS student_count,

    ROUND(AVG(f.score), 2) AS average_score,
    CAST(percentile_approx(f.score, 0.5) AS INT) AS median_score,
    MIN(f.score) AS min_score,
    MAX(f.score) AS max_score,

    MAX(f.gold_load_timestamp) AS gold_load_timestamp
FROM {catalog}.gold.fact_assessment_result f
LEFT JOIN {catalog}.gold.dim_batch b
    ON f.batch_key = b.batch_key
LEFT JOIN {catalog}.gold.dim_date d
    ON f.assessment_year_date_key = d.date_key
LEFT JOIN {catalog}.gold.dim_assessment_domain dom
    ON f.domain_key = dom.domain_key
LEFT JOIN {catalog}.gold.dim_proficiency_band band
    ON f.proficiency_band_key = band.proficiency_band_key
GROUP BY
    f.batch_id,
    b.batch_label,
    d.calendar_year,
    dom.domain,
    dom.domain_sort_order,
    band.proficiency_band,
    band.proficiency_band_sort_order,
    band.score_range_label
""")

display(
    spark.table(f"{catalog}.reporting.vw_assessment_by_domain")
    .orderBy(
        "batch_id",
        "assessment_year",
        "domain_sort_order",
        "proficiency_band_sort_order"
    )
)

batch_id,batch_label,assessment_year,domain,domain_sort_order,proficiency_band,proficiency_band_sort_order,score_range_label,assessment_count,student_count,average_score,median_score,min_score,max_score,gold_load_timestamp
2025-01-15,Batch 1 - 2024 data,2024,Reading,1,Low,1,250 to 399,3790,3790,341.64,347,250,399,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Reading,1,Medium,2,400 to 549,4150,4150,465.54,460,400,549,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Reading,1,High,3,550 and above,2060,2060,598.33,594,550,670,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Writing,2,Low,1,250 to 399,4332,4332,336.5,341,250,399,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Writing,2,Medium,2,400 to 549,4051,4051,467.22,462,400,549,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Writing,2,High,3,550 and above,1617,1617,593.04,590,550,655,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Numeracy,3,Low,1,250 to 399,4165,4165,338.21,342,250,399,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Numeracy,3,Medium,2,400 to 549,4049,4049,467.58,463,400,549,2026-06-08T05:37:27.731Z
2025-01-15,Batch 1 - 2024 data,2024,Numeracy,3,High,3,550 and above,1786,1786,594.3,591,550,660,2026-06-08T05:37:27.731Z
2026-01-15,Batch 2 - 2025 data,2025,Reading,1,Low,1,250 to 399,2620,2620,349.59,356,250,399,2026-06-08T05:37:49.261Z


In [0]:
display(
    spark.table(f"{catalog}.reporting.vw_assessment_by_school")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

display(
    spark.table(f"{catalog}.reporting.vw_assessment_by_domain")
    .groupBy("batch_id")
    .count()
    .orderBy("batch_id")
)

batch_id,count
2025-01-15,50
2026-01-15,48


batch_id,count
2025-01-15,9
2026-01-15,9


In [0]:
# Final Validation

from functools import reduce
from pyspark.sql import functions as F

reporting_views = [
    "vw_data_quality_rule_detail",
    "vw_data_quality_summary",
    "vw_defect_log",
    "vw_attendance_by_school_month",
    "vw_attendance_by_year_level",
    "vw_assessment_by_school",
    "vw_assessment_by_domain"
]

reporting_validation_dfs = []

for view_name in reporting_views:
    df = spark.table(f"{catalog}.reporting.{view_name}")

    validation_df = (
        df.groupBy("batch_id")
        .agg(F.count("*").alias("row_count"))
        .withColumn("reporting_view", F.lit(view_name))
        .select("reporting_view", "batch_id", "row_count")
    )

    reporting_validation_dfs.append(validation_df)

reporting_output_validation_df = reduce(
    lambda df1, df2: df1.unionByName(df2),
    reporting_validation_dfs
)

display(
    reporting_output_validation_df
    .orderBy("reporting_view", "batch_id")
)


reporting_view,batch_id,row_count
vw_assessment_by_domain,2025-01-15,9
vw_assessment_by_domain,2026-01-15,9
vw_assessment_by_school,2025-01-15,50
vw_assessment_by_school,2026-01-15,48
vw_attendance_by_school_month,2025-01-15,600
vw_attendance_by_school_month,2026-01-15,576
vw_attendance_by_year_level,2025-01-15,156
vw_attendance_by_year_level,2026-01-15,156
vw_data_quality_rule_detail,2025-01-15,11
vw_data_quality_rule_detail,2026-01-15,11
